# Combined SisFall + UMA Dataset Classification

This notebook combines both SisFall and UMA datasets to train a robust model for 5-class activity recognition:
- Walking
- Sitting
- Fall
- Stairs
- Jogging

## Dataset Integration Strategy
1. **SisFall Dataset**: Use processed features from existing pipeline
2. **UMA Dataset**: Extract compatible features and map activities
3. **Combined Training**: Train on both datasets with domain adaptation techniques

In [45]:
import pandas as pd
import numpy as np
import os
import glob
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline

# Feature extraction
from tsfresh import extract_features, select_features
from tsfresh.utilities.dataframe_functions import impute
from tsfresh.feature_extraction import ComprehensiveFCParameters, EfficientFCParameters

# Imbalanced learning
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

import joblib
import warnings
warnings.filterwarnings('ignore')

print("📦 All libraries imported successfully!")

📦 All libraries imported successfully!


## 1. UMA Activity Mapping Function

In [56]:
def map_uma_activity_to_target(activity_code):
    """Map UMA activity codes to our 5 target classes"""
    activity_mapping = {
        1: 'Walking',     # Walking
        2: 'Jogging',     # Jogging/Running  
        5: 'Stairs',      # GoDownstairs
        6: 'Stairs',      # GoUpstairs
        13: 'Fall',       # ForwardFall
        14: 'Fall',       # BackwardFall
        15: 'Fall',       # LateralFall
        # We'll need to find Sitting activities in UMA or exclude them
    }
    return activity_mapping.get(activity_code, None)

def get_uma_activity_name(activity_code):
    """Get UMA activity name for reference"""
    uma_activities = {
        1: 'Walking', 2: 'Running', 3: 'Bending', 4: 'LyingDown',
        5: 'GoDownstairs', 6: 'GoUpstairs', 7: 'JumpingUp', 
        8: 'Hopping', 9: 'SittingDown', 10: 'StandingUp',
        11: 'TurningRight', 12: 'TurningLeft', 13: 'ForwardFall',
        14: 'BackwardFall', 15: 'LateralFall'
    }
    return uma_activities.get(activity_code, f'Unknown_{activity_code}')

# Print UMA activity mapping for reference
print("UMA Activity Mapping to Target Classes:")
print("======================================")
for activity_code in range(1, 16):
    target_class = map_uma_activity_to_target(activity_code)
    activity_name = get_uma_activity_name(activity_code)
    status = "✅ INCLUDED" if target_class else "❌ EXCLUDED"
    print(f"{activity_code:2d}: {activity_name:15s} -> {target_class or 'None':10s} {status}")

UMA Activity Mapping to Target Classes:
 1: Walking         -> Walking    ✅ INCLUDED
 2: Running         -> Jogging    ✅ INCLUDED
 3: Bending         -> None       ❌ EXCLUDED
 4: LyingDown       -> None       ❌ EXCLUDED
 5: GoDownstairs    -> Stairs     ✅ INCLUDED
 6: GoUpstairs      -> Stairs     ✅ INCLUDED
 7: JumpingUp       -> None       ❌ EXCLUDED
 8: Hopping         -> None       ❌ EXCLUDED
 9: SittingDown     -> None       ❌ EXCLUDED
10: StandingUp      -> None       ❌ EXCLUDED
11: TurningRight    -> None       ❌ EXCLUDED
12: TurningLeft     -> None       ❌ EXCLUDED
13: ForwardFall     -> Fall       ✅ INCLUDED
14: BackwardFall    -> Fall       ✅ INCLUDED
15: LateralFall     -> Fall       ✅ INCLUDED


## 2. Load UMA Dataset

In [57]:
def load_uma_data(uma_folder='output_uma', max_files_per_activity=None):
    """
    Load UMA dataset with comprehensive activity mapping
    """
    print(f"📂 Loading UMA data from {uma_folder}...")
    
    uma_data = []
    uma_labels = []
    uma_ids = []
    activity_counts = defaultdict(int)
    
    # Get all subject folders
    subject_folders = glob.glob(os.path.join(uma_folder, 'Subject*'))
    
    for subject_folder in sorted(subject_folders):
        subject_name = os.path.basename(subject_folder)
        print(f"\n📋 Processing {subject_name}...")
        
        # Get all activity folders for this subject
        activity_folders = glob.glob(os.path.join(subject_folder, 'Activity*'))
        
        for activity_folder in sorted(activity_folders):
            activity_name = os.path.basename(activity_folder)
            activity_code = int(activity_name.replace('Activity', ''))
            
            # Map to target class
            target_class = map_uma_activity_to_target(activity_code)
            if target_class is None:
                continue  # Skip activities not in our target classes
            
            # Get all trial folders
            trial_folders = glob.glob(os.path.join(activity_folder, 'Trial*'))
            
            files_processed = 0
            for trial_folder in sorted(trial_folders):
                # Check max files limit
                if max_files_per_activity and activity_counts[target_class] >= max_files_per_activity:
                    break
                    
                # Find CSV file in trial folder
                csv_files = glob.glob(os.path.join(trial_folder, '*.csv'))
                if not csv_files:
                    continue
                    
                csv_file = csv_files[0]
                
                try:
                    # Load CSV data
                    df = pd.read_csv(csv_file)
                    
                    # Extract sensor data (columns 1-6: Acc X,Y,Z, Gyro X,Y,Z)
                    sensor_cols = [
                        'Accelerometer: x-axis (g)', 'Accelerometer: y-axis (g)', 'Accelerometer: z-axis (g)',
                        'Gyroscope: x-axis (rad/s)', 'Gyroscope: y-axis (rad/s)', 'Gyroscope: z-axis (rad/s)'
                    ]
                    
                    if all(col in df.columns for col in sensor_cols):
                        sensor_data = df[sensor_cols].values.T  # Shape: (6, n_samples)
                        
                        # Create unique ID
                        file_id = f"UMA_{subject_name}_{activity_name}_Trial{os.path.basename(trial_folder)[-1]}"
                        
                        uma_data.append(sensor_data)
                        uma_labels.append(target_class)
                        uma_ids.append(file_id)
                        
                        activity_counts[target_class] += 1
                        files_processed += 1
                        
                except Exception as e:
                    print(f"⚠️  Error processing {csv_file}: {e}")
                    continue
            
            if files_processed > 0:
                activity_desc = get_uma_activity_name(activity_code)
                print(f"   ✅ {activity_desc} -> {target_class}: {files_processed} files")
    
    print(f"\n📊 UMA Dataset Summary:")
    print(f"Total files loaded: {len(uma_data)}")
    print("\nClass distribution:")
    for class_name, count in sorted(activity_counts.items()):
        print(f"  {class_name}: {count}")
    
    return uma_data, uma_labels, uma_ids

# Load UMA data (limit files for faster processing during development)
uma_data, uma_labels, uma_ids = load_uma_data(
    uma_folder='output_uma', 
    max_files_per_activity=50  # Remove this limit for full dataset
)

📂 Loading UMA data from output_uma...

📋 Processing Subject1...
   ✅ Walking -> Walking: 3 files

📋 Processing Subject10...
   ✅ Walking -> Walking: 3 files
   ✅ GoDownstairs -> Stairs: 2 files
   ✅ GoUpstairs -> Stairs: 2 files

📋 Processing Subject11...
   ✅ Walking -> Walking: 3 files
   ✅ ForwardFall -> Fall: 6 files
   ✅ BackwardFall -> Fall: 6 files
   ✅ LateralFall -> Fall: 6 files
   ✅ Running -> Jogging: 3 files

📋 Processing Subject12...
   ✅ Walking -> Walking: 3 files
   ✅ GoDownstairs -> Stairs: 3 files
   ✅ GoUpstairs -> Stairs: 3 files

📋 Processing Subject13...
   ✅ Walking -> Walking: 3 files
   ✅ GoDownstairs -> Stairs: 2 files
   ✅ GoUpstairs -> Stairs: 2 files

📋 Processing Subject14...
   ✅ Walking -> Walking: 3 files
   ✅ ForwardFall -> Fall: 3 files
   ✅ BackwardFall -> Fall: 3 files
   ✅ LateralFall -> Fall: 3 files
   ✅ GoDownstairs -> Stairs: 3 files
   ✅ GoUpstairs -> Stairs: 3 files

📋 Processing Subject15...
   ✅ Walking -> Walking: 1 files
   ✅ ForwardFall

## 3. Load SisFall Dataset

In [48]:
def load_sisfall_features_and_labels():
    """
    Load pre-extracted SisFall features and create labels
    """
    print("📂 Loading SisFall features...")
    
    try:
        # Try to load the full feature set
        feature_files = [
            'extracted_features_full_Gyro_20250618_224358.csv',
            'extracted_features_full_Acc_20250618_192619.csv',
            'extracted_features_full_Acc_transformed.csv'
        ]
        
        features = None
        for feature_file in feature_files:
            if os.path.exists(feature_file):
                features = pd.read_csv(feature_file, index_col=0)
                print(f"✅ Loaded SisFall features from {feature_file}: {features.shape}")
                break
        
        if features is None:
            print("❌ No SisFall feature files found!")
            return None, None
        
        # Create labels from feature IDs using SisFall activity mapping
        id_label_pairs = []
        for idx in features.index:
            # SisFall activity code mapping to our 5 target classes
            if idx.startswith('D01_') or idx.startswith('D02_'):  # Walking slowly/quickly
                id_label_pairs.append((idx, 'Walking'))
            elif idx.startswith('D03_') or idx.startswith('D04_'):  # Jogging slowly/quickly
                id_label_pairs.append((idx, 'Jogging'))
            elif idx.startswith('D05_') or idx.startswith('D06_'):  # Stairs up/down
                id_label_pairs.append((idx, 'Stairs'))
            elif any(idx.startswith(f'D{i:02d}_') for i in [7, 8, 9, 10, 11, 12, 13]):  # Sitting activities
                id_label_pairs.append((idx, 'Sitting'))
            elif idx.startswith('F') or 'fall' in idx.lower():  # Fall activities
                id_label_pairs.append((idx, 'Fall'))
            else:
                # Default mapping for other activities
                print(f"⚠️  Unknown activity code in ID: {idx}")
                continue
        
        # Create labels DataFrame
        labels_df = pd.DataFrame(id_label_pairs, columns=['ID', 'Label'])
        
        # Get common IDs between features and labels
        common_ids = set(features.index) & set(labels_df['ID'])
        common_ids_list = list(common_ids)
        
        X_sisfall = features.loc[common_ids_list]
        y_sisfall = labels_df.set_index('ID').loc[common_ids_list]['Label']
        
        print(f"✅ Final SisFall dataset: {X_sisfall.shape}")
        print("SisFall class distribution:")
        class_counts = y_sisfall.value_counts()
        for class_name, count in class_counts.items():
            print(f"  {class_name}: {count}")
        
        return X_sisfall, y_sisfall
        
    except Exception as e:
        print(f"❌ Error loading SisFall data: {e}")
        return None, None

# Load SisFall features and labels
X_sisfall, y_sisfall = load_sisfall_features_and_labels()

📂 Loading SisFall features...
✅ Loaded SisFall features from extracted_features_full_Gyro_20250618_224358.csv: (6940, 4662)
✅ Final SisFall dataset: (6940, 4662)
SisFall class distribution:
  Walking: 1896
  Jogging: 1880
  Fall: 1798
  Sitting: 750
  Stairs: 616


## 4. Extract Features from UMA Data

In [49]:
def extract_manual_features(df, sensor_columns):
    """
    Fallback manual feature extraction when TSFresh fails
    """
    print("🔧 Extracting manual features...")
    
    features_dict = {}
    
    for sample_id in df['id'].unique():
        sample_data = df[df['id'] == sample_id]
        sample_features = {}
        
        for col in sensor_columns:
            values = sample_data[col].values
            
            # Basic statistical features
            sample_features[f'{col}_mean'] = np.mean(values)
            sample_features[f'{col}_std'] = np.std(values)
            sample_features[f'{col}_min'] = np.min(values)
            sample_features[f'{col}_max'] = np.max(values)
            sample_features[f'{col}_median'] = np.median(values)
            sample_features[f'{col}_var'] = np.var(values)
            sample_features[f'{col}_range'] = np.max(values) - np.min(values)
            
            # Additional features
            sample_features[f'{col}_q25'] = np.percentile(values, 25)
            sample_features[f'{col}_q75'] = np.percentile(values, 75)
            sample_features[f'{col}_iqr'] = np.percentile(values, 75) - np.percentile(values, 25)
            
        features_dict[sample_id] = sample_features
    
    # Convert to DataFrame
    features_df = pd.DataFrame.from_dict(features_dict, orient='index')
    print(f"Manual features extracted: {features_df.shape}")
    
    return features_df

def extract_features_from_uma_data(uma_data, uma_labels, uma_ids):
    """
    Extract TSFresh features from UMA sensor data
    """
    print("🔧 Extracting features from UMA data...")
    
    # Convert UMA data to TSFresh format
    all_dfs = []
    
    for idx, (data, label, file_id) in enumerate(zip(uma_data, uma_labels, uma_ids)):
        if idx % 50 == 0:
            print(f"Processing sample {idx+1}/{len(uma_data)}...")
        
        try:
            # data shape: (6, n_samples) -> (n_samples, 6)
            sensor_data = data.T
            
            # Create DataFrame for this sample
            time_series_length = sensor_data.shape[0]
            
            sample_df = pd.DataFrame({
                'id': [file_id] * time_series_length,
                'time': range(time_series_length),
                'Acc_X': sensor_data[:, 0],
                'Acc_Y': sensor_data[:, 1], 
                'Acc_Z': sensor_data[:, 2],
                'Gyro_X': sensor_data[:, 3],
                'Gyro_Y': sensor_data[:, 4],
                'Gyro_Z': sensor_data[:, 5],
                'Tag': [label] * time_series_length
            })
            
            all_dfs.append(sample_df)
            
        except Exception as e:
            print(f"⚠️  Error processing sample {idx} ({file_id}): {e}")
            continue
    
    if not all_dfs:
        print("❌ No valid UMA data processed!")
        return None, None
    
    # Combine all samples
    combined_df = pd.concat(all_dfs, ignore_index=True)
    print(f"✅ Combined UMA dataset shape: {combined_df.shape}")
    
    # Extract features using TSFresh
    print("🔧 Extracting TSFresh features...")
    
    # Use efficient parameters for faster processing
    extraction_settings = EfficientFCParameters()
    
    try:
        # Extract features for sensor columns
        sensor_columns = ['Acc_X', 'Acc_Y', 'Acc_Z', 'Gyro_X', 'Gyro_Y', 'Gyro_Z']
        
        # Verify input data types before TSFresh
        print("🔧 Verifying input data for TSFresh...")
        input_df = combined_df[['id', 'time'] + sensor_columns]
        print(f"Input data types: {input_df.dtypes}")
        print(f"Input shape: {input_df.shape}")
        print(f"Sample data:\n{input_df.head()}")
        
        # Ensure all sensor columns are numeric
        for col in sensor_columns:
            input_df[col] = pd.to_numeric(input_df[col], errors='coerce')
        
        # Remove any rows with all NaN sensor values
        input_df = input_df.dropna(subset=sensor_columns, how='all')
        print(f"After cleaning input shape: {input_df.shape}")
        
        try:
            features = extract_features(
                input_df, 
                column_id='id', 
                column_sort='time',
                default_fc_parameters=extraction_settings,
                n_jobs=1  # Use single process for stability
            )
        except Exception as tsfresh_error:
            print(f"⚠️  TSFresh extraction failed: {tsfresh_error}")
            print("🔄 Falling back to manual feature extraction...")
            
            # Fallback: Manual feature extraction
            features = extract_manual_features(input_df, sensor_columns)
        
        if features is None or features.empty:
            print("❌ No features extracted!")
            return None, None
        
        print(f"Raw extracted features shape: {features.shape}")
        print(f"Feature data types: {features.dtypes.value_counts()}")
        
        # Debug: Check for problematic data types
        problematic_cols = []
        for col in features.columns:
            if features[col].dtype == 'object' or features[col].dtype == 'string':
                problematic_cols.append(col)
                print(f"⚠️  Found object/string column: {col} with values: {features[col].unique()[:5]}")
        
        if problematic_cols:
            print(f"Found {len(problematic_cols)} problematic columns")
        
        # Handle data type issues before any operations
        print("🔧 Cleaning feature data types...")
        
        # Convert all columns to numeric, handling edge cases
        for col in features.columns:
            try:
                # First try direct conversion
                features[col] = pd.to_numeric(features[col], errors='coerce')
            except Exception as e:
                print(f"⚠️  Error converting column {col}: {e}")
                # Force conversion by replacing problematic values
                features[col] = features[col].astype(str)
                features[col] = pd.to_numeric(features[col], errors='coerce')
        
        # Check for infinite values and replace with NaN
        print("🔧 Handling infinite values...")
        try:
            features = features.replace([np.inf, -np.inf], np.nan)
        except Exception as e:
            print(f"⚠️  Error replacing infinite values: {e}")
            # Manual infinite value replacement
            for col in features.columns:
                features[col] = features[col].replace([np.inf, -np.inf], np.nan)
        
        # Handle NaN/inf values with manual imputation
        print("🔧 Handling missing values...")
        
        try:
            # Fill NaN values with column medians (more robust than mean)
            features = features.fillna(features.median())
        except Exception as e:
            print(f"⚠️  Error with median fillna: {e}")
            # Fallback: fill with 0
            features = features.fillna(0)
        
        # If there are still NaN values (all-NaN columns), fill with 0
        features = features.fillna(0)
        
        # Verify no NaN or inf values remain
        print("🔧 Verifying data cleanliness...")
        try:
            # Check data types first
            non_numeric_cols = []
            for col in features.columns:
                if not pd.api.types.is_numeric_dtype(features[col]):
                    non_numeric_cols.append(col)
                    print(f"⚠️  Non-numeric column found: {col} (dtype: {features[col].dtype})")
            
            if non_numeric_cols:
                print(f"Converting {len(non_numeric_cols)} non-numeric columns...")
                for col in non_numeric_cols:
                    features[col] = pd.to_numeric(features[col], errors='coerce').fillna(0)
            
            # Now safely check for NaN and inf values
            nan_count = features.isna().sum().sum()
            
            # Check for infinite values more safely
            inf_count = 0
            for col in features.columns:
                try:
                    col_inf = np.isinf(features[col]).sum()
                    inf_count += col_inf
                except Exception:
                    # If we can't check for inf, assume it's not numeric and convert
                    features[col] = pd.to_numeric(features[col], errors='coerce').fillna(0)
            
            print(f"Remaining NaN values: {nan_count}")
            print(f"Remaining infinite values: {inf_count}")
            
            if nan_count > 0 or inf_count > 0:
                print("⚠️  Still have problematic values, applying final cleanup...")
                features = features.fillna(0)
                features = features.replace([np.inf, -np.inf], 0)
                
        except Exception as e:
            print(f"⚠️  Error during verification: {e}")
            print("Applying aggressive cleanup...")
            # Aggressive cleanup
            for col in features.columns:
                features[col] = pd.to_numeric(features[col], errors='coerce').fillna(0)
            features = features.replace([np.inf, -np.inf], 0)
        
        print(f"✅ Extracted UMA features shape: {features.shape}")
        
        # Create labels dataframe
        labels_df = combined_df.groupby('id')['Tag'].first().reset_index()
        labels_df.columns = ['ID', 'Label']
        
        # Align features and labels
        common_ids = set(features.index) & set(labels_df['ID'])
        common_ids_list = list(common_ids)
        
        X_uma = features.loc[common_ids_list]
        y_uma = labels_df.set_index('ID').loc[common_ids_list]['Label']
        
        print(f"✅ Final UMA features: {X_uma.shape}")
        print("UMA class distribution:")
        for class_name, count in y_uma.value_counts().items():
            print(f"  {class_name}: {count}")
        
        return X_uma, y_uma
        
    except Exception as e:
        print(f"❌ Error extracting features: {e}")
        return None, None

# Extract features from UMA data
if uma_data:
    X_uma, y_uma = extract_features_from_uma_data(uma_data, uma_labels, uma_ids)
else:
    print("❌ No UMA data available for feature extraction")
    X_uma, y_uma = None, None

🔧 Extracting features from UMA data...
Processing sample 1/166...
Processing sample 51/166...
Processing sample 101/166...
Processing sample 151/166...
✅ Combined UMA dataset shape: (48702, 9)
🔧 Extracting TSFresh features...
🔧 Verifying input data for TSFresh...
Input data types: id        object
time       int64
Acc_X     object
Acc_Y     object
Acc_Z     object
Gyro_X    object
Gyro_Y    object
Gyro_Z    object
dtype: object
Input shape: (48702, 8)
Sample data:
                              id  time     Acc_X     Acc_Y     Acc_Z  \
0  UMA_Subject1_Activity1_Trial1     0 -0.740234  0.571289  0.249512   
1  UMA_Subject1_Activity1_Trial1     1 -0.842285  0.598145  0.297119   
2  UMA_Subject1_Activity1_Trial1     2 -0.833252  0.425781  0.332031   
3  UMA_Subject1_Activity1_Trial1     3 -0.721191  0.518311  0.116455   
4  UMA_Subject1_Activity1_Trial1     4 -0.741699  0.651611  0.380127   

     Gyro_X    Gyro_Y    Gyro_Z  
0 -0.176715  0.018817  0.319477  
1  1.524163  0.256073     0.63

## 5. Combine Datasets and Feature Alignment

In [50]:
def align_and_combine_datasets(X_sisfall, y_sisfall, X_uma, y_uma):
    """
    Align features between datasets and combine them
    """
    print("🔗 Aligning and combining datasets...")
    
    if X_sisfall is None or X_uma is None:
        print("❌ One or both datasets are missing!")
        return None, None, None, None
    
    print(f"SisFall features: {X_sisfall.shape}")
    print(f"UMA features: {X_uma.shape}")
    
    # Find common features
    common_features = list(set(X_sisfall.columns) & set(X_uma.columns))
    print(f"\n🎯 Common features found: {len(common_features)}")
    
    if len(common_features) == 0:
        print("❌ No common features found between datasets!")
        print("\nSisFall feature sample:", list(X_sisfall.columns[:5]))
        print("UMA feature sample:", list(X_uma.columns[:5]))
        return None, None, None, None
    
    # Select common features
    X_sisfall_aligned = X_sisfall[common_features]
    X_uma_aligned = X_uma[common_features]
    
    # Combine datasets
    X_combined = pd.concat([X_sisfall_aligned, X_uma_aligned], ignore_index=False)
    y_combined = pd.concat([y_sisfall, y_uma], ignore_index=False)
    
    # Create dataset source labels
    dataset_source = (['SisFall'] * len(X_sisfall_aligned) + 
                     ['UMA'] * len(X_uma_aligned))
    
    print(f"\n✅ Combined dataset shape: {X_combined.shape}")
    print("\n📊 Combined class distribution:")
    combined_counts = y_combined.value_counts()
    for class_name, count in combined_counts.items():
        sisfall_count = sum((y_sisfall == class_name).values) if class_name in y_sisfall.values else 0
        uma_count = sum((y_uma == class_name).values) if class_name in y_uma.values else 0
        print(f"  {class_name}: {count:4d} (SisFall: {sisfall_count:3d}, UMA: {uma_count:3d})")
    
    return X_combined, y_combined, dataset_source, common_features

# Combine datasets
X_combined, y_combined, dataset_source, common_features = align_and_combine_datasets(
    X_sisfall, y_sisfall, X_uma, y_uma
)

🔗 Aligning and combining datasets...
SisFall features: (6940, 4662)
UMA features: (166, 60)

🎯 Common features found: 0
❌ No common features found between datasets!

SisFall feature sample: ['Acc X__variance_larger_than_standard_deviation', 'Acc X__has_duplicate_max', 'Acc X__has_duplicate_min', 'Acc X__has_duplicate', 'Acc X__sum_values']
UMA feature sample: ['Acc_X_mean', 'Acc_X_std', 'Acc_X_min', 'Acc_X_max', 'Acc_X_median']


## 6. Data Preprocessing and Model Training

In [55]:
def train_combined_model(X_combined, y_combined, dataset_source):
    """
    Train a model on the combined dataset with proper validation
    """
    print("🤖 Training combined model...")
    
    if X_combined is None or y_combined is None:
        print("❌ No combined data available for training!")
        return None
    
    # Prepare data
    X = X_combined.copy()
    y = y_combined.copy()
    
    # Handle any remaining NaN values
    print(f"🔧 Handling missing values...")
    X = X.fillna(X.median())
    
    # Split data ensuring both datasets are represented in train/test
    print(f"📊 Splitting data...")
    
    # Stratified split to maintain class balance
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    print(f"Training set: {X_train.shape}")
    print(f"Test set: {X_test.shape}")
    
    # Training set class distribution
    print("\n📊 Training set class distribution:")
    for class_name, count in y_train.value_counts().items():
        print(f"  {class_name}: {count}")
    
    # Create and train pipeline with SMOTE and Random Forest
    print(f"\n🚀 Training Random Forest with SMOTE...")
    
    pipeline = ImbPipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=42, k_neighbors=3)),
        ('rf', RandomForestClassifier(
            n_estimators=200,
            max_depth=15,
            min_samples_split=5,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1
        ))
    ])
    
    # Train the model
    pipeline.fit(X_train, y_train)
    
    # Make predictions
    y_pred = pipeline.predict(X_test)
    
    # Calculate accuracy
    accuracy = accuracy_score(y_test, y_pred)
    print(f"\n🎯 Test Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    
    # Detailed classification report
    print("\n📋 Classification Report:")
    print(classification_report(y_test, y_pred))
    
    # Cross-validation
    print("\n🔄 Cross-validation (5-fold):")
    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='accuracy')
    print(f"CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")
    
    return {
        'model': pipeline,
        'X_train': X_train,
        'X_test': X_test,
        'y_train': y_train,
        'y_test': y_test,
        'y_pred': y_pred,
        'accuracy': accuracy,
        'cv_scores': cv_scores
    }

# Train the combined model
if X_combined is not None:
    results = train_combined_model(X_combined, y_combined, dataset_source)
else:
    print("❌ Cannot train model - no combined data available")
    results = None

❌ Cannot train model - no combined data available


## 7. Model Evaluation and Visualization

In [52]:
def visualize_results(results):
    """
    Create visualizations for model performance
    """
    if results is None:
        print("❌ No results to visualize")
        return
    
    print("📊 Creating visualizations...")
    
    # Set up the plotting style
    plt.style.use('default')
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Combined SisFall + UMA Model Performance', fontsize=16, fontweight='bold')
    
    # 1. Confusion Matrix
    cm = confusion_matrix(results['y_test'], results['y_pred'])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.unique(results['y_test']))
    disp.plot(ax=axes[0,0], cmap='Blues', values_format='d')
    axes[0,0].set_title('Confusion Matrix')
    
    # 2. Class Distribution Comparison
    train_counts = results['y_train'].value_counts()
    test_counts = results['y_test'].value_counts()
    
    x = np.arange(len(train_counts))
    width = 0.35
    
    axes[0,1].bar(x - width/2, train_counts.values, width, label='Train', alpha=0.8)
    axes[0,1].bar(x + width/2, test_counts.values, width, label='Test', alpha=0.8)
    axes[0,1].set_xlabel('Activity Class')
    axes[0,1].set_ylabel('Number of Samples')
    axes[0,1].set_title('Train/Test Class Distribution')
    axes[0,1].set_xticks(x)
    axes[0,1].set_xticklabels(train_counts.index, rotation=45)
    axes[0,1].legend()
    axes[0,1].grid(axis='y', alpha=0.3)
    
    # 3. Cross-validation scores
    cv_scores = results['cv_scores']
    axes[1,0].bar(range(1, len(cv_scores)+1), cv_scores, alpha=0.7, color='green')
    axes[1,0].axhline(y=cv_scores.mean(), color='red', linestyle='--', 
                     label=f'Mean: {cv_scores.mean():.3f}')
    axes[1,0].set_xlabel('Fold')
    axes[1,0].set_ylabel('Accuracy')
    axes[1,0].set_title('Cross-Validation Scores')
    axes[1,0].legend()
    axes[1,0].grid(axis='y', alpha=0.3)
    
    # 4. Feature Importance (Top 15)
    if hasattr(results['model'].named_steps['rf'], 'feature_importances_'):
        feature_importance = results['model'].named_steps['rf'].feature_importances_
        feature_names = results['X_train'].columns
        
        # Get top 15 features
        top_indices = np.argsort(feature_importance)[-15:]
        top_importance = feature_importance[top_indices]
        top_names = [feature_names[i] for i in top_indices]
        
        # Truncate feature names for better display
        top_names_short = [name[:30] + '...' if len(name) > 30 else name for name in top_names]
        
        axes[1,1].barh(range(len(top_importance)), top_importance, alpha=0.7)
        axes[1,1].set_yticks(range(len(top_importance)))
        axes[1,1].set_yticklabels(top_names_short, fontsize=8)
        axes[1,1].set_xlabel('Feature Importance')
        axes[1,1].set_title('Top 15 Most Important Features')
        axes[1,1].grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Summary statistics
    print("\n" + "="*60)
    print("📈 COMBINED MODEL PERFORMANCE SUMMARY")
    print("="*60)
    print(f"Test Accuracy: {results['accuracy']:.4f} ({results['accuracy']*100:.2f}%)")
    print(f"CV Mean Accuracy: {results['cv_scores'].mean():.4f} ± {results['cv_scores'].std():.4f}")
    print(f"Training Samples: {len(results['y_train'])}")
    print(f"Test Samples: {len(results['y_test'])}")
    print(f"Number of Features: {results['X_train'].shape[1]}")

# Visualize results
if results:
    visualize_results(results)

## 8. Save Model and Generate Report

In [53]:
def save_model_and_report(results, common_features):
    """
    Save the trained model and generate a comprehensive report
    """
    if results is None:
        print("❌ No results to save")
        return
    
    print("💾 Saving model and generating report...")
    
    # Save the model
    model_filename = 'combined_sisfall_uma_model.pkl'
    joblib.dump(results['model'], model_filename)
    print(f"✅ Model saved as {model_filename}")
    
    # Save feature list
    feature_filename = 'combined_model_features.json'
    import json
    with open(feature_filename, 'w') as f:
        json.dump(common_features, f, indent=2)
    print(f"✅ Feature list saved as {feature_filename}")
    
    # Generate detailed report
    report_filename = 'COMBINED_MODEL_REPORT.md'
    
    report_content = f"""# Combined SisFall + UMA Model Report

Generated on: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}

## Model Overview

This model combines data from both SisFall and UMA datasets to perform 5-class activity recognition:
- Walking
- Sitting  
- Fall
- Stairs
- Jogging

## Dataset Statistics

- **Total Training Samples**: {len(results['y_train'])}
- **Total Test Samples**: {len(results['y_test'])}
- **Number of Features**: {results['X_train'].shape[1]}
- **Common Features Used**: {len(common_features)}

### Class Distribution (Training Set)

| Class | Count |
|-------|-------|
"""
    
    for class_name, count in results['y_train'].value_counts().items():
        report_content += f"| {class_name} | {count} |\n"
    
    report_content += f"""

## Model Performance

- **Test Accuracy**: {results['accuracy']:.4f} ({results['accuracy']*100:.2f}%)
- **Cross-Validation Mean**: {results['cv_scores'].mean():.4f} ± {results['cv_scores'].std():.4f}

### Cross-Validation Scores

"""
    
    for i, score in enumerate(results['cv_scores'], 1):
        report_content += f"- Fold {i}: {score:.4f}\n"
    
    report_content += f"""

## Classification Report

```
{classification_report(results['y_test'], results['y_pred'])}
```

## Model Configuration

- **Algorithm**: Random Forest with SMOTE
- **Preprocessing**: StandardScaler + SMOTE
- **Random Forest Parameters**:
  - n_estimators: 200
  - max_depth: 15
  - min_samples_split: 5
  - min_samples_leaf: 2

## Top Features

The most important features for classification:

"""
    
    if hasattr(results['model'].named_steps['rf'], 'feature_importances_'):
        feature_importance = results['model'].named_steps['rf'].feature_importances_
        feature_names = results['X_train'].columns
        
        # Get top 10 features
        top_indices = np.argsort(feature_importance)[-10:]
        
        for i in reversed(top_indices):
            report_content += f"- {feature_names[i]}: {feature_importance[i]:.6f}\n"
    
    report_content += f"""

## Usage Instructions

To use this model:

```python
import joblib
import pandas as pd

# Load the model
model = joblib.load('{model_filename}')

# Load feature list
import json
with open('{feature_filename}', 'r') as f:
    required_features = json.load(f)

# Make predictions
# X_new should have the same features as required_features
predictions = model.predict(X_new[required_features])
```

## Notes

- This model was trained on combined SisFall and UMA datasets
- SMOTE was used to handle class imbalance
- Feature alignment was performed to ensure compatibility between datasets
- The model uses TSFresh-extracted features from sensor data
"""
    
    with open(report_filename, 'w') as f:
        f.write(report_content)
    
    print(f"✅ Report saved as {report_filename}")
    print("\n🎉 Model training and evaluation complete!")

# Save model and generate report
if results and common_features:
    save_model_and_report(results, common_features)

## 9. Test Model with Individual Dataset Performance

In [54]:
def test_individual_dataset_performance(results, dataset_source):
    """
    Test model performance on individual datasets (SisFall vs UMA)
    """
    if results is None or dataset_source is None:
        print("❌ Cannot test individual performance - missing data")
        return
    
    print("🔍 Testing individual dataset performance...")
    
    # Get dataset source for test set
    test_indices = results['X_test'].index
    sisfall_indices = [i for i, idx in enumerate(test_indices) if idx in X_sisfall.index]
    uma_indices = [i for i, idx in enumerate(test_indices) if idx in X_uma.index]
    
    print(f"\nTest set composition:")
    print(f"SisFall samples: {len(sisfall_indices)}")
    print(f"UMA samples: {len(uma_indices)}")
    
    if len(sisfall_indices) > 0:
        # SisFall performance
        y_test_sisfall = results['y_test'].iloc[sisfall_indices]
        y_pred_sisfall = results['y_pred'][sisfall_indices]
        
        sisfall_accuracy = accuracy_score(y_test_sisfall, y_pred_sisfall)
        print(f"\n📊 SisFall Test Performance:")
        print(f"Accuracy: {sisfall_accuracy:.4f} ({sisfall_accuracy*100:.2f}%)")
        print("\nSisFall Classification Report:")
        print(classification_report(y_test_sisfall, y_pred_sisfall))
    
    if len(uma_indices) > 0:
        # UMA performance  
        y_test_uma = results['y_test'].iloc[uma_indices]
        y_pred_uma = results['y_pred'][uma_indices]
        
        uma_accuracy = accuracy_score(y_test_uma, y_pred_uma)
        print(f"\n📊 UMA Test Performance:")
        print(f"Accuracy: {uma_accuracy:.4f} ({uma_accuracy*100:.2f}%)")
        print("\nUMA Classification Report:")
        print(classification_report(y_test_uma, y_pred_uma))
    
    # Cross-dataset performance comparison
    print("\n" + "="*50)
    print("📈 DATASET-SPECIFIC PERFORMANCE SUMMARY")
    print("="*50)
    if len(sisfall_indices) > 0:
        print(f"SisFall Accuracy: {sisfall_accuracy:.4f} ({sisfall_accuracy*100:.2f}%)")
    if len(uma_indices) > 0:
        print(f"UMA Accuracy: {uma_accuracy:.4f} ({uma_accuracy*100:.2f}%)")
    print(f"Combined Accuracy: {results['accuracy']:.4f} ({results['accuracy']*100:.2f}%)")

# Test individual dataset performance
if results and 'X_test' in results:
    test_individual_dataset_performance(results, dataset_source)